# Outlier Analysis — Template Notebook

Use this notebook to investigate outliers in raw datasets before running the ETL pipeline.

## Overview

This notebook provides a structured approach to detect, analyze, and document outliers in your dataset using multiple statistical methods:

- **IQR (Interquartile Range)**: Robust to extreme values
- **Z-Score**: Standard deviations from the mean
- **Modified Z-Score**: MAD-based, resistant to outliers

## When to Use This Notebook

- **Before ETL**: Identify data quality issues early
- **During EDA**: Understand data distribution and anomalies
- **When detecting drift**: Compare outlier patterns across time

See `README_outlier_notebook.md` for detailed instructions.

## Setup

Configure your dataset path and target column below.

In [ ]:
# template_outlier_analysis.ipynb
# Outlier Analysis — Template Notebook
# ====================================
# Use this notebook to investigate outliers in raw datasets
# before running the ETL pipeline.

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import HTML, display
from energizados.eda._outlier_detector import OutlierDetector
from energizados.eda.utils import classify_columns

# Configuration
DATASET_PATH = "../data/raw/sample_dataset.parquet"  # Change this
TARGET_COL = "target"  # Optional — set to None if no target

# Set style
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)

print("✅ Setup complete")

## Load Data

Load the dataset and display basic information.

In [ ]:
# Load dataset
df = pd.read_parquet(DATASET_PATH)
print(f"Shape: {df.shape}")
print(f"\nColumns: {list(df.columns)}")

# Basic info
print("\n--- Dataset Info ---")
df.info()

# First few rows
print("\n--- First 5 Rows ---")
display(df.head())

## Classify Columns

Automatically classify columns into numeric, categorical, and consumption types.

In [ ]:
# Classify columns automatically
col_types = classify_columns(df)
numeric_cols = col_types.get("numeric", [])
categorical_cols = col_types.get("categorical", [])
consumption_cols = col_types.get("consumption", [])

print(f"📊 Column Classification:")
print(f"  - Numeric: {len(numeric_cols)} columns")
print(f"  - Categorical: {len(categorical_cols)} columns")
print(f"  - Consumption: {len(consumption_cols)} columns")

if numeric_cols:
    print(f"\nNumeric columns: {numeric_cols}")
if consumption_cols:
    print(f"\nConsumption columns: {consumption_cols}")

## Multi-Method Outlier Detection

Detect outliers using multiple statistical methods for robustness.

In [ ]:
# Detect outliers using multiple methods
detector = OutlierDetector(
    methods=["iqr", "zscore", "modified_zscore"],
    iqr_multiplier=1.5,
    zscore_threshold=3.0,
)

outlier_summary = {}
for col in numeric_cols:
    outlier_summary[col] = detector.detect(df[col])
    print(f"✓ {col}: IQR outliers = {outlier_summary[col]['iqr']['outlier_pct']:.2f}%")

print(f"\n📈 Detected outliers for {len(numeric_cols)} numeric columns")

## Summary Table

Compare outlier counts across different methods.

In [ ]:
# Build summary table
summary_df = pd.DataFrame([
    {
        "column": col,
        "iqr_count": r["iqr"]["outlier_count"],
        "iqr_pct": r["iqr"]["outlier_pct"],
        "zscore_count": r["zscore"]["outlier_count"],
        "zscore_pct": r["zscore"]["outlier_pct"],
    }
    for col, r in outlier_summary.items()
]).sort_values("iqr_pct", ascending=False)

print("=== Outlier Summary Table ===")
print(summary_df.to_string(index=False))

# Display as DataFrame for better formatting
display(summary_df.style.background_gradient(subset=['iqr_pct'], cmap='Reds'))

## Boxplots

Visualize outliers with boxplots (only showing columns with outliers).

In [ ]:
# Boxplots with outlier markers
from energizados.eda.plots import plot_outlier_boxplots

# Only show columns with > 0% outliers for cleaner viz
cols_with_outliers = [c for c, r in outlier_summary.items() 
                      if r.get("iqr", {}).get("outlier_pct", 0) > 0]

if cols_with_outliers:
    print(f"📊 Creating boxplots for {len(cols_with_outliers)} columns with outliers...")
    
    # Prepare outlier masks
    outlier_masks = {}
    for col in cols_with_outliers:
        outlier_masks[col] = outlier_summary[col]["iqr"].get("outlier_mask", pd.Series(False, index=df.index))
    
    svg_dict = plot_outlier_boxplots(df, cols_with_outliers[:6], outlier_masks)
    
    # Display first few plots
    for i, (col, svg) in enumerate(list(svg_dict.items())[:3]):
        print(f"\n=== {col} ===")
        display(HTML(svg))
else:
    print("✅ No columns with outliers detected")

## Consumption Anomalies

Analyze consumption-specific outlier patterns.

In [ ]:
# Analyze consumption-specific patterns
if consumption_cols:
    from energizados.eda.plots import plot_consumption_anomalies
    
    print(f"📊 Analyzing {len(consumption_cols)} consumption columns...")
    svg_dict = plot_consumption_anomalies(df, consumption_cols, TARGET_COL)
    
    # Display first few periods
    for period, svg in list(svg_dict.items())[:3]:
        print(f"\n=== {period} ===")
        display(HTML(svg))
else:
    print("ℹ️  No consumption columns detected")

## Row-level Analysis

Identify rows that are outliers across multiple columns.

In [ ]:
# Which rows are outliers across multiple columns?
outlier_counts_per_row = pd.Series(0, index=df.index)
for col in numeric_cols:
    mask = outlier_summary[col].get("iqr", {}).get("outlier_mask", pd.Series(False, index=df.index))
    outlier_counts_per_row += mask.astype(int)

df["outlier_score"] = outlier_counts_per_row

print("=== Outlier Score Distribution ===")
print(df["outlier_score"].describe())

# Show most outlier-prone rows
top_outliers = df.nlargest(10, "outlier_score")
print(f"\nTop 10 outlier-prone rows (out of {len(df)} total):")

# Select columns to display
display_cols = ["outlier_score"] + numeric_cols[:5]
if TARGET_COL and TARGET_COL in df.columns:
    display_cols.append(TARGET_COL)

display(top_outliers[display_cols])

## Recommendations

Data preprocessing recommendations based on outlier analysis.

In [ ]:
# Print preprocessing recommendations
print("=== PREPROCESSING RECOMMENDATIONS ===\n")

for col in outlier_summary:
    pct = outlier_summary[col]["iqr"]["outlier_pct"]
    
    if pct > 20:
        status = "⚠️ HIGH — Consider capping (winsorizing) or removing column"
        action = "winsorize"
    elif pct > 10:
        status = "🔍 MEDIUM — Investigate origin before deciding"
        action = "investigate"
    elif pct > 5:
        status = "🔍 LOW-MEDIUM — Monitor in next data refresh"
        action = "monitor"
    else:
        status = "✅ OK — Within acceptable range"
        action = "none"
    
    print(f"{col}: {pct:.1f}% outliers — {status}")
    print(f"   Recommended action: {action}\n")

# Summary
high_outlier_cols = [c for c, r in outlier_summary.items() if r['iqr']['outlier_pct'] > 20]
if high_outlier_cols:
    print(f"\n⚠️  {len(high_outlier_cols)} columns with high outlier percentage:")
    print(f"   {high_outlier_cols}")

## Export Report

Save outlier analysis results as JSON for documentation.

In [ ]:
# Export outlier report as JSON
import json
from datetime import datetime

report = {
    "generated_at": str(datetime.now()),
    "dataset": DATASET_PATH,
    "n_rows": len(df),
    "n_numeric_cols": len(numeric_cols),
    "n_categorical_cols": len(categorical_cols),
    "n_consumption_cols": len(consumption_cols),
    "target_column": TARGET_COL,
    "outlier_summary": {
        col: {k: v for k, v in r.items() if k != "outlier_mask"}
        for col, r in outlier_summary.items()
    },
}

output_file = "outlier_report.json"
with open(output_file, "w") as f:
    json.dump(report, f, indent=2, default=str)

print(f"✅ Report saved to {output_file}")
print(f"   {len(outlier_summary)} columns analyzed")

# Display file info
import os
file_size = os.path.getsize(output_file)
print(f"   File size: {file_size / 1024:.1f} KB")

## Next Steps

After completing this analysis:

1. **Review high-outlier columns**: Investigate data collection or entry errors
2. **Apply preprocessing**: Use winsorization, capping, or removal based on recommendations
3. **Update ETL pipeline**: Add preprocessing steps to handle identified issues
4. **Document decisions**: Keep a record of how outliers were handled
5. **Re-run analysis**: After preprocessing, re-run to validate improvements

### Related Resources

- EDA documentation: `docs/eda.md`
- Full EDA report: Run `energizados run eda`
- Configuration: `config/eda.yaml`